# Fidelity Experiments — T4 Colab

Runs KV cache reuse fidelity evaluation on a T4 GPU.

**What this measures:** Whether splicing cached KV from one prompt prefix into
another's computation changes the generated output. Uses `DynamicCache.crop()`.

**Cell order:**
1. Install + setup
2. Upload package
3. Run GPT-2 float16 + float32
4. Run Phi-3 float16 + float32 (float32 uses `--dtype float32 --device cuda`)
5. Evaluate
6. Download

In [ ]:
# Install deps
!pip install -q torch transformers datasets accelerate huggingface-hub rouge-score numpy pyarrow pandas
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Upload & extract
from google.colab import files
print('Upload fidelity_t4_package.zip')
uploaded = files.upload()
import zipfile, os
with zipfile.ZipFile('fidelity_t4_package.zip', 'r') as z:
    z.extractall('fidelity_pkg')
os.chdir('fidelity_pkg')
print('Extracted.')

## GPT-2
Float16 (GPU) then Float32 (CPU). ~3 min each.

In [ ]:
!python run_fidelity_t4.py --device cuda:0 --models gpt2 --datasets samsum alpaca_eval banking77 daily_dialog ag_news --n_samples 10 --max_gen_tokens 64 --output_dir ../fidelity_gpt2_f16 2>&1

In [ ]:
!python run_fidelity_t4.py --device cpu --models gpt2 --datasets samsum alpaca_eval banking77 daily_dialog ag_news --n_samples 10 --max_gen_tokens 64 --output_dir ../fidelity_gpt2_f32 2>&1

## Phi-3 Mini
Float16 (GPU) then Float32 on GPU (uses CUDA + float32 dtype to avoid CPU OOM). ~5-10 min each.

In [ ]:
!python run_fidelity_t4.py --device cuda:0 --models phi3mini --datasets samsum alpaca_eval banking77 daily_dialog ag_news --n_samples 10 --max_gen_tokens 64 --output_dir ../fidelity_phi3_f16 2>&1

In [ ]:
!python run_fidelity_t4.py --device cuda --dtype float32 --models phi3mini --datasets samsum alpaca_eval banking77 daily_dialog ag_news --n_samples 10 --max_gen_tokens 64 --output_dir ../fidelity_phi3_f32 2>&1

## Evaluate

In [ ]:
from rouge_score import rouge_scorer
import json, os, glob
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
for d in sorted(glob.glob('../fidelity_*')):
    f = os.path.join(d, 'all_results.json')
    if not os.path.exists(f): continue
    r = json.load(open(f))
    matches = sum(1 for x in r if x['ref_text'] == x['reuse_text'])
    print(f'\n{d.split("_")[-1].upper()}: {matches}/{len(r)} ({matches/len(r)*100:.1f}%)')
    for mk in sorted(set(x['model'] for x in r)):
        items = [x for x in r if x['model'] == mk]
        m = sum(1 for x in items if x['ref_text'] == x['reuse_text'])
        print(f'  {mk:>10}: {m}/{len(items)} ({m/len(items)*100:.1f}%)')

## Download

In [ ]:
from google.colab import files
for d in ['fidelity_gpt2_f16','fidelity_gpt2_f32','fidelity_phi3_f16','fidelity_phi3_f32']:
    !zip -r /content/{d}.zip /content/{d}
    files.download(f'/content/{d}.zip')